In [1]:
import sys
sys.path.append("../src")

import pandas as pd

from pdf_extractor import extract_text_from_pdf
from recommender import recommend_jobs
from evaluator import precision_at_k, recall_at_k

C:\Users\Asus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7865.23it/s]


In [2]:
jobs_path = "../data/raw/jobs/jobs_matching_cv_categories.csv"

jobs_df = pd.read_csv(jobs_path)

print("Liczba ofert:", len(jobs_df))
jobs_df.head()

Liczba ofert: 25


,Job Title,Description
0,Accountant,"Looking for Accountant with accounting, payrol..."
1,Banking Analyst,"Banking Analyst required with finance, risk an..."
2,Business Development Specialist,"Business Development Specialist with sales, ne..."
3,Chef,"Chef position requiring kitchen management, fo..."
4,Construction Project Manager,"Construction Project Manager with AutoCAD, bud..."


In [3]:
test_cases = [
    {
        "cv_path": "../data/raw/cvs/resume_dataset/data/data/TEACHER/10504237.pdf",
        "expected_keywords": ["teacher", "education"]
    },
    {
        "cv_path": "../data/raw/cvs/resume_dataset/data/data/CHEF/10588874.pdf",
        "expected_keywords": ["chef"]
    },
    {
        "cv_path": "../data/raw/cvs/resume_dataset/data/data/HR/11698189.pdf",
        "expected_keywords": ["hr", "recruitment"]
    }
]

In [4]:
results = []

for case in test_cases:
    cv_text = extract_text_from_pdf(case["cv_path"])

    recommended_jobs = recommend_jobs(
        cv_text,
        jobs_df,
        top_n=5
    )

    precision = precision_at_k(
        recommended_jobs,
        case["expected_keywords"],
        k=5
    )

    recall = recall_at_k(
        recommended_jobs,
        case["expected_keywords"],
        k=5
    )

    results.append({
        "CV": case["cv_path"],
        "Expected": case["expected_keywords"],
        "Precision@5": precision,
        "Recall@5": recall,
        "Top jobs": recommended_jobs["Job Title"].tolist()
    })

evaluation_df = pd.DataFrame(results)

evaluation_df

,CV,Expected,Precision@5,Recall@5,Top jobs
0,../data/raw/cvs/resume_dataset/data/data/TEACH...,"[teacher, education]",0.2,0.5,"[Teacher, Agriculture Specialist, Mechanical E..."
1,../data/raw/cvs/resume_dataset/data/data/CHEF/...,[chef],0.2,1.0,"[Chef, Sales Manager, Fitness Trainer, Constru..."
2,../data/raw/cvs/resume_dataset/data/data/HR/11...,"[hr, recruitment]",0.2,0.5,"[Sales Manager, HR Specialist, Fitness Trainer..."
